In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# 데이터베이스 내에 넣을 데이터프레임 가공 -> 저장

# 순서 - master_crop_variety -> map_region_weather_station -> weather_daily -> factor_external -> fact_trade

In [ ]:
import pandas as pd
import os
import re
import urllib.parse
from dotenv import load_dotenv
from datetime import datetime, date
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

load_dotenv()

In [ ]:
# 사전 DB 세팅 # 외부 세팅인 첨부한 .env 설정파일 참고해서 env 설정하자
# DB 정보
user = os.getenv("DB_USER")  # .env 파일에 DB_USER 설정해도 됨
host = os.getenv("DB_HOST")
password = os.getenv("DB_PW")  # DB_pw → 대소문자 주의 (env 키명)
password  = urllib.parse.quote_plus(password)
port = int(os.getenv("DB_PORT"))
db = os.getenv("DB_NAME")
engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{db}?charset=utf8mb4"
)

# 거래데이터 삽입_daily

In [ ]:
dtype_spec = {
    'gds_lclsf_cd': str,
    'gds_mclsf_cd': str,
    'gds_sclsf_cd': str,
    'plor_cd': str
}

df = pd.read_csv('data/유통공사_도매시장_감자_20180101-20250531.csv', encoding='cp949', dtype=dtype_spec)
# df2 = pd.read_csv('data/유통공사_도매시장_무_20200101-20250531.csv', encoding='cp949', dtype=dtype_spec)
# df3 = pd.read_csv('data/유통공사_무_retry_성공_20250713_153852.csv', encoding='cp949', dtype=dtype_spec)
# df4 = pd.read_csv('data/상추/상추_실패3.csv', encoding='cp949', dtype=dtype_spec)

# 이후 코드는 그대로 실행합니다.
df = pd.concat([df1, df2, df3], axis=0)
df.drop_duplicates(inplace=True)


In [ ]:
# plor_cd가 문자열이 아닐 가능성 대비
df['plor_cd'] = df['plor_cd'].fillna('').astype(str)

pattern = r'^[^0-9]+$'

condition = (
    df['totprc'].isna() | (df['totprc'] <= 0) |
    df['unit_tot_qty'].isna() | (df['unit_tot_qty'] <= 0) |
    df['plor_cd'].str.strip().isin(['0', '0.0']) |
    df['plor_cd'].str.match(pattern, na=False) |
    df['plor_nm'].isna() |
    (df['plor_nm'] == 0)
)

# 조건에 해당하는 행 추출
df_filtered = df[~condition]

# 수입산을 하나로 몰까 했지만.. 그냥 패스
# df.loc[df['plor_cd'].str.startswith('800')]['plor_cd'] = '800000'

# 직팜코드 테이블 소환 - 향후 이걸 DB로 가져오자
df_region = pd.read_csv('data/map_region_weather_station_utf-8.csv', encoding='utf-8')

# 직팜코드 붙이기
df_region['plor_cd'] = df_region['plor_cd'].astype(str)
df_merged_1 = pd.merge(df_filtered, df_region[['plor_cd', 'j_sanji_cd']],  on='plor_cd', how='left')
df_merged_1['trd_clcln_ymd'] = pd.to_datetime(df_merged_1['trd_clcln_ymd'], format='%Y-%m-%d')

# 아이템 풀코드 장착
for col in ['gds_lclsf_cd', 'gds_mclsf_cd', 'gds_sclsf_cd']:
    df_merged_1[col] = df_merged_1[col].astype(str) 
    df_merged_1[col] = df_merged_1[col].str.zfill(2)
df_merged_1['crop_full_code'] = df_merged_1['gds_lclsf_cd']+df_merged_1['gds_mclsf_cd']+df_merged_1['gds_sclsf_cd']

# df_trade['item_code'] = df_trade['crop_full_code'].str[:4]

# 필요한 열만 선별

df = df_merged_1[['trd_clcln_ymd', 'crop_full_code', 'j_sanji_cd', 'unit_tot_qty', 'totprc']]

df['item_code'] = df['crop_full_code'].str[:4]
df = df.dropna()
df.info()
df.head()

In [ ]:
# groupby 리스트에 'item_code'를 추가합니다.
df_merged = df.groupby(['trd_clcln_ymd', 'crop_full_code', 'item_code', 'j_sanji_cd']).sum(['unit_tot_qty', 'totprc']).reset_index()

df_merged['avg_prc'] = round(df_merged['totprc'] / df_merged['unit_tot_qty'])
df_merged

In [ ]:
# 등급 라벨링

# j_sanji_cd <= 2000 인 값만 필터링
mask_domestic = df_merged['j_sanji_cd'] < 2000

# trd_clcln_ymd 기준으로 그룹화하여 각 그룹별 avg_prc의 80%, 20% 분위 계산
quantiles = df_merged[mask_domestic].groupby('trd_clcln_ymd')['avg_prc'].quantile([0.2, 0.8]).unstack()

# 함수 정의: trd_clcln_ymd와 avg_prc 기준으로 '고', '중', '저' 구분
def assign_grade(row):
    if row['j_sanji_cd'] > 2000:
        return '수입'
    q20 = quantiles.loc[row['trd_clcln_ymd'], 0.2]
    q80 = quantiles.loc[row['trd_clcln_ymd'], 0.8]
    if row['avg_prc'] >= q80:
        return '고'
    elif row['avg_prc'] >= q20:
        return '중'
    else:
        return '저'

# grade_label 컬럼 생성
df_merged['grade_label'] = df_merged.apply(assign_grade, axis=1)
df_merged

In [ ]:
df_merged.drop(columns='avg_prc', inplace=True)

In [ ]:
df

In [ ]:
# df_merged.to_csv('datasets/fact_trade_사과.csv', encoding='utf-8', index=False)

In [ ]:
# df_merged = pd.read_csv('datasets/fact_trade.csv', encoding='utf-8')

In [ ]:
# 데이터 베이스 저장
# to_sql로 insert (테이블명, 커넥션, 옵션)
df_merged.to_sql(
    name='fact_trade',    # 실제 DB의 테이블명
    con=engine,
    if_exists='append',            # append: 추가 / replace: 전체 덮어쓰기
    index=False
)

print("DB 적재 완료!")

# 거래데이터 삽입_weekly

In [ ]:
# # sqlalchemy에서 text 함수를 import 합니다.
# from sqlalchemy import text

# ITEM = "무"  # 맨 아래 백업 파일명에 들어갈 작물명!
# query = """
# SELECT *
# FROM fact_trade
# WHERE crop_full_code NOT LIKE '0601%%'
#     AND crop_full_code NOT LIKE '1101%%'
#     AND crop_full_code NOT LIKE '1201%%'
#     AND crop_full_code NOT LIKE '1001%%'
#     AND crop_full_code NOT LIKE '1101%%'
    
    
# """

# # 'with' 구문을 사용하여 connection을 명시적으로 생성하고 전달합니다.
# with engine.connect() as connection:
#     df_trade = pd.read_sql(text(query), connection)

# # with 블록이 끝나면 connection은 자동으로 닫힙니다.
# df_trade.info()

In [ ]:
from sqlalchemy import text

# item_code가 '1001'인 데이터를 선택하도록 쿼리 수정
query = """
SELECT *
FROM fact_trade
WHERE item_code = '0501'
"""

# 'with' 구문을 사용하여 connection을 명시적으로 생성하고 전달합니다.
with engine.connect() as connection:
    df_trade = pd.read_sql(text(query), connection)

# with 블록이 끝나면 connection은 자동으로 닫힙니다.
df_trade.info()

In [ ]:
df_trade.sample(5)

In [ ]:
%%time
# 1작물 할때 20초 정도 걸림! 참고!
##주간 병합하기 
# 주차 표기
def get_week_of_year(date):
    date = pd.to_datetime(date)
    year = date.year
    week_number = date.isocalendar().week
    return f"{year}{week_number:02d}"

df_trade['weekno'] = df_trade['trd_clcln_ymd'].apply(get_week_of_year)

# 데이터 머지 (일, 작물코드, 산지)
df_trade.drop(columns='grade_label', inplace=True)
df_merged = df_trade.groupby(["weekno", "crop_full_code", "item_code", "j_sanji_cd", "year_part"]).sum(['unit_tot_qty','totprc']).reset_index()

# 주간 평균 삽입
df_merged['avg_prc'] = round(df_merged['totprc'] / df_merged['unit_tot_qty'])

## 등급 라벨링

# j_sanji_cd != 2000 인 값만 필터링
mask_domestic = df_merged['j_sanji_cd'] != '2000'

# trd_clcln_ymd 기준으로 그룹화하여 각 그룹별 avg_prc의 80%, 20% 분위 계산
quantiles = df_merged[mask_domestic].groupby('weekno')['avg_prc'].quantile([0.2, 0.8]).unstack()

# 함수 정의: trd_clcln_ymd와 avg_prc 기준으로 '고', '중', '저' 구분
def assign_grade(row):
    if row['j_sanji_cd'] == '2000':
        return '수입'
    q20 = quantiles.loc[row['weekno'], 0.2]
    q80 = quantiles.loc[row['weekno'], 0.8]
    if row['avg_prc'] >= q80:
        return '고'
    elif row['avg_prc'] >= q20:
        return '중'
    else:
        return '저'

# grade_label 컬럼 생성
df_merged['grade_label'] = df_merged.apply(assign_grade, axis=1)
df_merged.drop_duplicates()
df_merged.drop(columns='year_part')

# 체크 및 필터링 (결측치`)
num_cols = ['totprc', 'unit_tot_qty', 'avg_prc']
for col in num_cols:
    df_merged[col] = pd.to_numeric(df_merged[col], errors='coerce')
    
df_merged.to_csv(f'data/fact_trade_weekly_{ITEM}_BACKUP.csv', encoding='cp949', index=False)

df_merged.info()
df_merged.head()

In [ ]:
# 데이터 베이스 저장
# to_sql로 insert (테이블명, 커넥션, 옵션)
df_merged.to_sql(
    name='fact_trade_weekly',    # 실제 DB의 테이블명
    con=engine,
    if_exists='append',            # append: 추가 / replace: 전체 덮어쓰기
    index=False
)

print("DB 적재 완료!")